In [1]:
import glob
import os

import time
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import seaborn as sns
import holidays

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import lightgbm as lgb
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

import rasterio
import math
import rasterio
from rasterio.mask import mask
from shapely.geometry import Point
from shapely.ops import transform
from pyproj import Transformer

import warnings
warnings.filterwarnings("ignore")


In [5]:
BASE_DIR              = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()

PATH_EXTRA_DATA       = os.path.join(BASE_DIR, "../../data/extra/")
PATH_TRAFFIC          = os.path.join(BASE_DIR, "../../data/traffic/")
FILEPATH_RICH         = os.path.join(BASE_DIR, "../../data/richtingen.csv")
FILEPATH_SITE         = os.path.join(BASE_DIR, "../../data/sites.csv")
FILEPATH_INFRA        = os.path.join(BASE_DIR, "output/sites_enriched_clean.csv")

FILEPATH_POP_TIF      = PATH_EXTRA_DATA + "bel_pop_2025_CN_100m_R2025A_v1.tif"
FILEPATH_WEATHER      = PATH_EXTRA_DATA + 'site_id_weather_output_new.csv'

TRAFFIC_COLS  = ['site_id', 'direction', 'type', 'from', 'to', 'count']
RICH_COLS     = ['site_id', 'direction', 'name']
SITE_COLS     = ['site_id', 'site_nr', 'longtitude', 'latitude', 'name', 'domain', 'road_nbr','district_nbr', 'municipality', 'interval', 'date_installed']

# ── PREDICTION CONFIG ─────────────────────────────────────────────────────────
PREDICT_DATE          = "2026-05-20"          # ← change only this to predict a different date

FILEPATH_LEUVEN_SITE  = os.path.join(BASE_DIR, "output/leuven_sites.csv")
FILEPATH_LEUVEN_INFRA = os.path.join(BASE_DIR, "output/leuven_bike_roads.csv")
FILEPATH_MODEL        = os.path.join(BASE_DIR, "../yenha/model_final.pickle")

# Output paths derived automatically from PREDICT_DATE
_date_tag        = PREDICT_DATE.replace("-", "_")
OUTPUT_PRED_PATH = os.path.join(BASE_DIR, f"explo/andy/output/leuven_predictions_{_date_tag}.csv")

print(f"Predict date  : {PREDICT_DATE}")
print(f"Output path   : {OUTPUT_PRED_PATH}")
print(f"Leuven sites  : {FILEPATH_LEUVEN_SITE}")
print(f"Model path    : {FILEPATH_MODEL}")

Predict date  : 2026-05-20
Output path   : c:\Users\Admin\Desktop\KU Leuven\Sem 2\MDA\mda_project\mda_project\explo\andy\explo/andy/output/leuven_predictions_2026_05_20.csv
Leuven sites  : c:\Users\Admin\Desktop\KU Leuven\Sem 2\MDA\mda_project\mda_project\explo\andy\output/leuven_sites.csv
Model path    : c:\Users\Admin\Desktop\KU Leuven\Sem 2\MDA\mda_project\mda_project\explo\andy\../yenha/model_final.pickle


In [3]:
######## FEATURE ENGINEERING ########
def compute_neighbor_traffic_lag_hour_features(df_traffic, df_site):
    RADII_M = [200,500,1000,2000]
    RADII_KM = [r / 1000 for r in RADII_M]   # [0.2, 0.5, 1.0, 2.0]

    sites = df_site.copy()
    sites = sites[["site_id", "latitude", "longtitude"]].drop_duplicates("site_id").copy()
    sites = sites.sort_values("site_id").reset_index(drop=True)

    ids = sites["site_id"].to_numpy()
    lat = sites["latitude"].to_numpy()
    lon = sites["longtitude"].to_numpy()

    # Haversine for all pairs (vectorized)
    R = 6371.0
    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)
    dlat = lat_rad[:, None] - lat_rad[None, :]
    dlon = lon_rad[:, None] - lon_rad[None, :]
    a = (np.sin(dlat / 2) ** 2
        + np.cos(lat_rad)[:, None] * np.cos(lat_rad)[None, :] * np.sin(dlon / 2) ** 2)
    dist = 2 * R * np.arcsin(np.sqrt(a))

    # Fill diagonal with infinity so that sites don't consider themselves as neighbors
    np.fill_diagonal(dist, np.inf)

    dist_df = pd.DataFrame(dist, index=ids, columns=ids)

    # radius_maps[radius_m][site_id] = [list site within that radius]
    radius_maps = {r_m: {} for r_m in RADII_M}
    for site in ids:
        ordered = dist_df.loc[site].sort_values()
        for r_m, r_km in zip(RADII_M, RADII_KM):
            radius_maps[r_m][site] = ordered[ordered <= r_km].index.tolist()


    df_traffic_cleaned = df_traffic.copy()
    df_traffic_cleaned = df_traffic_cleaned[df_traffic_cleaned["type"] == "FIETSERS"]
    df_traffic_cleaned['count'] = df_traffic_cleaned['count'].fillna(0)
    df_traffic_cleaned["site_id"]   = df_traffic_cleaned["site_id"].astype("int32")
    df_traffic_cleaned["from"] = pd.to_datetime(df_traffic_cleaned["from"])
    df_traffic_cleaned["datehour"] = pd.to_datetime(df_traffic_cleaned["from"]).dt.floor("h")

    df_traffic_hourly = (
        df_traffic_cleaned
        .groupby(["site_id", "datehour"], as_index=False)["count"]
        .sum()
        .rename(columns={'count':'traffic_total'})
        .reset_index(drop=True)
    )
    traffic_wide = df_traffic_hourly.pivot_table(index="datehour",columns="site_id",values="traffic_total",aggfunc="sum").sort_index()

    lag_dfs = []
    for LAG in [1, 24, 168]:
        traffic_wide_lag = traffic_wide.shift(LAG)
        nan_template = pd.Series(np.nan, index=traffic_wide_lag.index)
        parts = []
        for site in ids:
            row_dict = {"site_id": site, "datehour": traffic_wide_lag.index}
            for r_m in RADII_M:
                rad_cols = [s for s in radius_maps[r_m][site] if s in traffic_wide_lag.columns]
                prefix = f"nb_r{r_m}m_lag{LAG}h"
                if rad_cols:
                    sub = traffic_wide_lag[rad_cols]
                    row_dict[f"{prefix}_mean"] = sub.mean(axis=1).values
                    row_dict[f"{prefix}_max"]  = sub.max(axis=1).values
                    row_dict[f"{prefix}_sum"]  = sub.sum(axis=1).values
                    row_dict[f"{prefix}_std"]  = sub.std(axis=1).values
                else:
                    for stat in ("mean", "max", "sum", "std"):
                        row_dict[f"{prefix}_{stat}"] = nan_template.values
            parts.append(pd.DataFrame(row_dict))
        lag_dfs.append(pd.concat(parts, ignore_index=True))

    df_fts_neighbor_lag = lag_dfs[0]
    for df_lag in lag_dfs[1:]:
        df_fts_neighbor_lag = df_fts_neighbor_lag.merge(
            df_lag.drop(columns=["site_id", "datehour"], errors="ignore")
            if False else df_lag,  # merge full second df
            on=["site_id", "datehour"],
            how="outer",
        )
    df_fts_neighbor_lag.columns = [col.replace('168h', '7d') for col in df_fts_neighbor_lag.columns]
    df_fts_neighbor_lag = df_fts_neighbor_lag[['site_id','datehour'] + [x for x in df_fts_neighbor_lag if x.startswith('nb')]]
    df_fts_neighbor_lag = df_fts_neighbor_lag.loc[:, df_fts_neighbor_lag.isna().mean() <= 0.7]

    return df_fts_neighbor_lag

def compute_neighbor_traffic_lag_15m_features(df_traffic, df_site):
    RADII_M = [200,500,1000,2000]
    RADII_KM = [r / 1000 for r in RADII_M]

    sites = df_site.copy()
    sites = sites[["site_id", "latitude", "longtitude"]].drop_duplicates("site_id").copy()
    sites = sites.sort_values("site_id").reset_index(drop=True)

    ids = sites["site_id"].to_numpy()
    lat = sites["latitude"].to_numpy()
    lon = sites["longtitude"].to_numpy()

    # Haversine for all pairs (vectorized)
    R = 6371.0
    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)
    dlat = lat_rad[:, None] - lat_rad[None, :]
    dlon = lon_rad[:, None] - lon_rad[None, :]
    a = (np.sin(dlat / 2) ** 2
        + np.cos(lat_rad)[:, None] * np.cos(lat_rad)[None, :] * np.sin(dlon / 2) ** 2)
    dist = 2 * R * np.arcsin(np.sqrt(a))

    # Fill diagonal with infinity so that sites don't consider themselves as neighbors
    np.fill_diagonal(dist, np.inf)
    dist_df = pd.DataFrame(dist, index=ids, columns=ids)

    # radius_maps[radius_m][site_id] = [list site within that radius]
    radius_maps = {r_m: {} for r_m in RADII_M}
    for site in ids:
        ordered = dist_df.loc[site].sort_values()
        for r_m, r_km in zip(RADII_M, RADII_KM):
            radius_maps[r_m][site] = ordered[ordered <= r_km].index.tolist()


    df_traffic_cleaned = df_traffic.copy()
    df_traffic_cleaned = df_traffic_cleaned[df_traffic_cleaned["type"] == "FIETSERS"]
    df_traffic_cleaned['count'] = df_traffic_cleaned['count'].fillna(0)
    df_traffic_cleaned["site_id"]   = df_traffic_cleaned["site_id"].astype("int32")
    df_traffic_cleaned["from"] = pd.to_datetime(df_traffic_cleaned["from"])
    df_traffic_cleaned["datehour"] = pd.to_datetime(df_traffic_cleaned["from"]).dt.floor("h")


    # Step 1: 15-min FIETSERS traffic, summed across directions 
    df_traffic_fiets_15min = (
        df_traffic_cleaned
        .groupby(["site_id", "from"], as_index=False)["count"]
        .sum()
        .rename(columns={"count": "total_fiets_15min"})
    )

    # Step 2: Wide matrix (rows = 15-min timestamp, cols = site_id)
    traffic_wide_15min = df_traffic_fiets_15min.pivot_table(
        index="from",
        columns="site_id",
        values="total_fiets_15min",
        aggfunc="sum",
    ).sort_index()
    traffic_wide_15min.index.name = "ts"

    # Step 3: Shift by 1 period (15 min) → previous 15-min traffic 
    # Row at ts=T in shifted table = original traffic at ts=(T-15min)
    traffic_wide_15min_shifted = traffic_wide_15min.shift(1)

    # Step 4: Keep only on-the-hour rows to align with datehour 
    # Row at ts=14:00 (minute==0) after shift = traffic from 13:45-14:00
    traffic_wide_prev15 = traffic_wide_15min_shifted[
        traffic_wide_15min_shifted.index.minute == 0
    ].copy()
    traffic_wide_prev15.index.name = "datehour"


    # Step 5: For each site and datehour, compute neighbor features based on previous 15-min traffic
    parts_15m = []
    nan_tpl = pd.Series(np.nan, index=traffic_wide_prev15.index)

    for site in ids:
        row = {"site_id":  site, "datehour": traffic_wide_prev15.index,}

        for r_m in RADII_M:
            rad_cols = [s for s in radius_maps[r_m][site] if s in traffic_wide_prev15.columns]
            pfx = f"nb_r{r_m}m_lag15m"

            if rad_cols:
                sub = traffic_wide_prev15[rad_cols]
                row[f"{pfx}_mean"] = sub.mean(axis=1).values
                row[f"{pfx}_max"]  = sub.max(axis=1).values
                row[f"{pfx}_sum"]  = sub.sum(axis=1).values
                row[f"{pfx}_std"]  = sub.std(axis=1).values
            else:
                for stat in ("mean", "max", "sum", "std"):
                    row[f"{pfx}_{stat}"] = nan_tpl.values
        parts_15m.append(pd.DataFrame(row))

    df_fts_neighbor_lag15m = pd.concat(parts_15m, ignore_index=True)
    df_fts_neighbor_lag15m = df_fts_neighbor_lag15m[['site_id','datehour'] + [x for x in df_fts_neighbor_lag15m if x.startswith('nb')]]
    df_fts_neighbor_lag15m = df_fts_neighbor_lag15m.loc[:, df_fts_neighbor_lag15m.isna().mean() <= 0.7]

    return df_fts_neighbor_lag15m
 
def compute_weather_features(df_weather, lags_h = [1, 2, 3, 6]):
    df = df_weather.copy()
    df["datehour"] = pd.to_datetime(df["datetime"])
    df = df.drop(columns=['latitude', 'longitude', 'datetime']).rename(columns={x: f"wt_{x}" for x in df.columns if x not in ['site_id', 'latitude', 'longitude','datehour']})


    df["datehour"] = pd.to_datetime(df["datehour"])
    df = df.sort_values(["site_id", "datehour"])

    grp = df.groupby("site_id")

    # ── Plain lags
    lag_cols = ["wt_precipitation", "wt_rain", "wt_temperature_2m", "wt_wind_speed_10m", "wt_cloud_cover", "wt_snowfall"]
    for col in lag_cols:
        if col not in df.columns:
            continue
        for lag in lags_h:
            df[f"{col}_lag{lag}h"] = grp[col].shift(lag)

    # ── Rolling aggregations (sum & mean, using only past values)
    rolling_windows = {
        "wt_precipitation": [3, 6, 12],   # accumulated rain last N hours
        "wt_temperature_2m": [3, 24],      # temp trend
    }
    for col, windows in rolling_windows.items():
        if col not in df.columns:
            continue
        for w in windows:
            # shift(1) → exclude current hour, use only past
            base = grp[col].shift(1)
            df[f"{col}_rolling_sum_{w}h"]  = base.transform(
                lambda x: x.rolling(w, min_periods=1).sum()
            )
            df[f"{col}_rolling_mean_{w}h"] = base.transform(
                lambda x: x.rolling(w, min_periods=1).mean()
            )

    # ── Derived lag features
    if "wt_temperature_2m" in df.columns:
        # Temperature trend: warming or cooling?
        df["wt_temp_change_1h"] = df["wt_temperature_2m"] - grp["wt_temperature_2m"].shift(1)
        df["wt_temp_change_3h"] = df["wt_temperature_2m"] - grp["wt_temperature_2m"].shift(3)

    if "wt_precipitation" in df.columns:
        # Binary: was it raining in each of the last 3 hours?
        for lag in [1, 2, 3]:
            df[f"wt_was_raining_lag{lag}h"] = (
                grp["wt_precipitation"].shift(lag) > 0.5
            ).astype(int)

        # Consecutive rainy hours (deterrence buildup)
        is_raining = (grp["wt_precipitation"].shift(1) > 0.5).astype(int)
        df["wt_consec_rain_hours"] = (
            is_raining
            .groupby(df["site_id"])
            .transform(lambda x: x * (x.groupby((x != x.shift()).cumsum()).cumcount() + 1))
        )

        # Flag: first dry hour after rain (pent-up demand spike)
        df["wt_first_dry_after_rain"] = (
            (grp["wt_precipitation"].shift(1) > 0.5) &   # was raining 1h ago
            (df["wt_precipitation"] <= 0.5)               # now dry
        ).astype(int)

    return df

def compute_population_features(df_site, tif_path):
    def population_estimate(lat, lon, radius_m, tif_path):
        with rasterio.open(tif_path) as src:
            METRIC_CRS = "EPSG:3035"  
            # 1. Covert (lat, lon) to m for accurate buffering
            to_metric = Transformer.from_crs("EPSG:4326", METRIC_CRS, always_xy=True)
            x_m, y_m = to_metric.transform(lon, lat)

            # 2. Create circle with radius_m meters
            circle = Point(x_m, y_m).buffer(radius_m)

            # 3. Transform circle to raster's CRS for masking
            to_raster = Transformer.from_crs(METRIC_CRS, src.crs, always_xy=True)
            circle_in_raster = transform(to_raster.transform, circle)

            # 4. Cut raster by circle, mask NoData automatically
            out, _ = mask(src, [circle_in_raster], crop=True, filled=False)

            total_pop = float(out.sum())                       # total population in the circle
            area_km2  = math.pi * (radius_m ** 2) / 1e6        # area of the circle in km²
            density   = total_pop / area_km2                   # population density in people/km²
            return total_pop, density, area_km2

    rows = []
    for loc in df_site.itertuples():
        name, lat, lon = loc.site_id, loc.latitude, loc.longtitude
        row = {"site_id": name, "latitude": lat, "longitude": lon}
        for r in (500, 1000, 5000):
            pop, dens, area = population_estimate(lat, lon, r, tif_path)
            row[f"pop_{r}m"] = round(pop, 1)
            row[f"density_{r}m"] = round(dens, 1)
        rows.append(row)

    df_fts_population = pd.DataFrame(rows)
    df_fts_population = df_fts_population[['site_id'] + [x for x in df_fts_population if x.startswith('pop') or x.startswith('density')]]
    return df_fts_population

def compute_datetime_features(df_traffic):
    """Calendar + cyclical features at hourly resolution (no 15-min slot)."""
    df_traffic_cleaned = df_traffic.copy()
    df_traffic_cleaned = df_traffic_cleaned.query('type=="FIETSERS"')
    df_traffic_cleaned["from"] = pd.to_datetime(df_traffic_cleaned["from"])
    df_traffic_cleaned["datehour"] = (pd.to_datetime(df_traffic_cleaned["from"].dt.strftime("%Y-%m-%d")) + pd.to_timedelta(df_traffic_cleaned["from"].dt.hour, unit="h"))
    df_traffic_cleaned["datehour"] = df_traffic_cleaned["from"].dt.floor("h")

    dt = df_traffic_cleaned[["site_id", "datehour"]].drop_duplicates().copy()
    dt["year"]      = dt["datehour"].dt.year
    dt["month"]     = dt["datehour"].dt.month
    dt["day"]       = dt["datehour"].dt.day
    dt["hour"]      = dt["datehour"].dt.hour
    dt["dayofweek"] = dt["datehour"].dt.dayofweek

    # Weekday / weekend
    dt["dt_is_weekend"] = dt["dayofweek"].isin([5, 6]).astype(int)
    dt["dt_is_weekday"] = 1 - dt["dt_is_weekend"]

    # Specific days
    dt["dt_is_monday"] = (dt["dayofweek"] == 0).astype(int)
    dt["dt_is_friday"] = (dt["dayofweek"] == 4).astype(int)

    # Hour-of-day buckets
    dt["dt_is_morning_rush"]   = dt["hour"].isin([7, 8, 9]).astype(int)
    dt["dt_is_afternoon_rush"] = dt["hour"].isin([16, 17, 18]).astype(int)
    dt["dt_is_rush_hour"]      = (dt["dt_is_morning_rush"] | dt["dt_is_afternoon_rush"]).astype(int)
    dt["dt_is_lunch_hour"]     = dt["hour"].isin([12, 13]).astype(int)
    dt["dt_is_night"]          = dt["hour"].isin([0, 1, 2, 3, 4, 5]).astype(int)
    dt["dt_is_business_hours"] = (dt["hour"].between(9, 17) & (dt["dt_is_weekday"] == 1)).astype(int)

    # Belgian pulic holiday
    bel_holiday_dates = list(holidays.Belgium(years=range(2024, 2027)).keys())
    dt["dt_is_holiday"] = dt["datehour"].dt.normalize().isin(bel_holiday_dates).astype(int)

    df2 = dt[['site_id','datehour', "year", "month", "day", "hour", "dayofweek"] + [x for x in dt if x.startswith('dt')]]

    return df2

def compute_site_features(df_traffic, df_site):
    df_traffic_cleaned = df_traffic.copy()
    df_traffic_cleaned = df_traffic_cleaned.query('type=="FIETSERS"')
    df_traffic_cleaned["from"] = pd.to_datetime(df_traffic_cleaned["from"])
    df_traffic_cleaned["datehour"] = (pd.to_datetime(df_traffic_cleaned["from"].dt.strftime("%Y-%m-%d")) + pd.to_timedelta(df_traffic_cleaned["from"].dt.hour, unit="h"))
    df_traffic_cleaned["datehour"] = df_traffic_cleaned["from"].dt.floor("h")


    sites = df_site.copy()
    sites["install_date"] = pd.to_datetime(sites["date_installed"], errors="coerce")
    sites["site_is_road_tunnel"]   = sites["road_nbr"].astype(str).str.upper().str.startswith("T", na=False).astype(int)
    sites["site_is_road_national"] = sites["road_nbr"].astype(str).str.upper().str.startswith("N", na=False).astype(int)
    sites["site_is_road_ring"]     = sites["road_nbr"].astype(str).str.upper().str.startswith("R", na=False).astype(int)
    sites["site_is_road_motorway"] = sites["road_nbr"].astype(str).str.upper().str.startswith("A", na=False).astype(int)

    df_fts_site = (
        df_traffic_cleaned[["site_id", "datehour"]]
        .drop_duplicates()
        .merge(
            sites[["site_id", "install_date"] + [x for x in sites if x.startswith('site_is')]],
            on="site_id", how="left",
        )
    )

    df_fts_site["site_sensor_age"] = ((df_fts_site["datehour"] - df_fts_site["install_date"]).dt.days / 365)
    df_fts_site = df_fts_site[["site_id", "datehour"] + [x for x in df_fts_site if x.startswith('site_is')]]

    return df_fts_site
    
def compute_infra_features(df_infra, df_site):
    df_infra00 = df_infra.copy() 
    df_site00 = df_site.copy()

    df_infra00 = df_infra00.rename(columns={'site_id':'site_nr'}).merge(df_site00[['site_id','site_nr']], on='site_nr', how='left')
    df_fts_infra = df_infra00.drop(columns=['sensor_age_days','has_cycleway'])

    poi_types = ['shop', 'education', 'hotel', 'hospital']
    for dist in ['250m', '500m', '1000m']:
        cols = [f'poi_{typ}_{dist}' for typ in poi_types]
        cols_present = [c for c in cols if c in df_fts_infra.columns]
        df_fts_infra[f'poi_{dist}'] = df_fts_infra[cols_present].sum(axis=1)

    df_fts_infra['road_category'] = df_fts_infra['road_category_en'].replace({
        'not_applicable': 'others',
        'high_capacity_road': 'others',
        'expressway_limited_access': 'others',
    })

    road_category_dummies = pd.get_dummies(df_fts_infra['road_category'], prefix='road_category', dtype=int)
    df_fts_infra = pd.concat([df_infra00, road_category_dummies], axis=1)
    df_fts_infra = df_fts_infra.rename(columns={c: f'road_{c}' for c in ['length_m', 'dist_to_segment_m', 'bike_lane_width_m', 'has_cycleway', 'bike_width_imputed']})
    df_fts_infra = df_fts_infra[['site_id'] + [x for x in df_fts_infra if (x.startswith('road') or x.startswith('poi')) and pd.api.types.is_numeric_dtype(df_fts_infra[x])]]

    return df_fts_infra

######## MODELING ########
def split_traintest(df, list_features, target):
    d = df.copy()
    X = d[list_features]
    y = d[target].astype(float)
    return X, y

def score_predictions(y_true, pred):
    pred = np.clip(pred, 0, None)
    return {
        "MAE":  mean_absolute_error(y_true, pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, pred)),
    }

def run_models(dmodels, 
               X_train, y_train, 
               X_val, y_val, 
               X_test, y_test):
    """
    Train and evaluate all models in dmodels, return dict_model with results.
    Optionally takes df_train/df_valid/FEATURE_COLS/TARGET for custom split for HGB.
    """
    dict_model = {}
    for name, model in dmodels.items():
        t0 = time.time()
        # Handle model fit logic
        if "HistGradientBoosting" in name:
            model.fit(X_train, y_train)

        elif "XGBoost" in name:
            model.fit(
                X_train, y_train, 
                eval_set=[(X_val, y_val)], 
                verbose=False 
            )

        elif "LightGBM" in name:
            model.fit(
                X_train, y_train,
                eval_set=[(X_val, y_val)],
                callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False), lgb.log_evaluation(period=0)]
            )
        else:
            model.fit(X_train, y_train)

        pred = model.predict(X_test)
        scores = score_predictions(y_test, pred)

        if name == "HistGradientBoosting":
            best_iter = model.n_iter_
        else:
            best_iter = getattr(model, "best_iteration_", getattr(model, "best_iteration", "N/A"))
        print(f"{name:20s} - {round(scores['MAE'],4)} - {best_iter}")

        dict_model[name] = {
            "model": model,
            "mae": scores["MAE"],
            "rmse": scores["RMSE"],
            "time": time.time() - t0,
            "best_iteration": best_iter
        }
    return dict_model

def stepwise_feature_selection_lgb(
    mode,
    list_feature_all,
    base_model,
    X_train,
    y_train,
    X_valid,
    y_valid,
    model_params=None,
):
    if model_params is None:
        model_params = {}
    if mode not in ["add", "remove"]:
        raise ValueError("mode must be either 'add' or 'remove'")
    model_class = lgb.LGBMRegressor
    log_dict = {}

    # 1. Extract initial features from the base model
    if hasattr(base_model, "feature_names_in_"):
        current_features = list(base_model.feature_names_in_)
    else:
        raise AttributeError("base_model must have .feature_names_in_ attribute.")

    # 2. Establish initial baseline (on validation set)
    X_valid_base = X_valid[current_features]
    base_preds = base_model.predict(X_valid_base)
    curr_mae = mean_absolute_error(y_valid, base_preds)

    log_dict['model_base'] = {
        'features': list(current_features),
        'MAE': curr_mae,
        'model': base_model
    }

    # 3. Determine candidates based on mode
    if mode == "add":
        candidates = [f for f in list_feature_all if f not in current_features]
    else:  # remove
        candidates = list(current_features)

    step_iter = 1

    # 4. Single pass through candidates
    for feat in candidates:
        # Construct the feature set
        if mode == "add":
            try_features = current_features + [feat]
        else:  # remove
            if len(current_features) <= 1:
                break
            try_features = [f for f in current_features if f != feat]
        
        # Train and evaluate on training/validation splits
        model_try = model_class(**model_params)
        model_try.fit(X_train[try_features], y_train)
        preds_try = model_try.predict(X_valid[try_features])
        mae_try = mean_absolute_error(y_valid, preds_try)

        action = "Add" if mode == "add" else "Remove"
        print(f"{action}: {feat:50s} | MAE: {mae_try:.5f} | current MAE: {curr_mae:.5f}")

        # If MAE improves (decreases), update current set and baseline
        if mae_try < curr_mae:
            status = "ADDED" if mode == "add" else "REMOVED"
            print(f"  => [{status}]: {feat:50s} | New Base MAE: {mae_try:.5f}")
            
            current_features = try_features
            curr_mae = mae_try

            log_dict[f'model_{mode}_{step_iter}'] = {
                'features': list(current_features),
                'MAE': curr_mae,
                'model': model_try,
                'feature_changed': feat
            }
            step_iter += 1

    return current_features, log_dict

######## CROSS-SITE NEIGHBOR FUNCTIONS (memory-optimised) ########

def compute_neighbor_traffic_lag_hour_features_cross(
        df_traffic, df_site_real, df_site_new, predict_date=PREDICT_DATE):
    """
    Compute hourly neighbor-traffic lag features for NEW sites using REAL sensor
    traffic.  Only the 24 hours of `predict_date` are returned, keeping memory use
    proportional to (N_new x 24) instead of (N_new x full_history).
    """
    import numpy as _np
    import pandas as _pd

    RADII_M  = [200, 500, 1000, 2000]
    RADII_KM = [r / 1000 for r in RADII_M]

    # ── real sites ────────────────────────────────────────────────────────────
    real = (df_site_real[["site_id","latitude","longtitude"]]
            .drop_duplicates("site_id").sort_values("site_id"))
    real_ids = real["site_id"].to_numpy()
    real_lat  = real["latitude"].to_numpy()
    real_lon  = real["longtitude"].to_numpy()

    # ── new (Leuven) sites ────────────────────────────────────────────────────
    new = (df_site_new[["site_id","latitude","longtitude"]]
           .drop_duplicates("site_id").sort_values("site_id"))
    new_ids = new["site_id"].to_numpy()
    new_lat  = new["latitude"].to_numpy()
    new_lon  = new["longtitude"].to_numpy()

    # ── haversine cross-distance (N_new × N_real) in km ──────────────────────
    R = 6371.0
    nl = _np.radians(new_lat);  nlo = _np.radians(new_lon)
    rl = _np.radians(real_lat); rlo = _np.radians(real_lon)
    dlat = nl[:, None] - rl[None, :]
    dlon = nlo[:, None] - rlo[None, :]
    a    = _np.sin(dlat/2)**2 + _np.cos(nl)[:,None]*_np.cos(rl)[None,:]*_np.sin(dlon/2)**2
    dist = 2 * R * _np.arcsin(_np.sqrt(a))
    dist_df = _pd.DataFrame(dist, index=new_ids, columns=real_ids)

    # ── radius lookup ─────────────────────────────────────────────────────────
    radius_maps = {r_m: {} for r_m in RADII_M}
    for site in new_ids:
        ordered = dist_df.loc[site].sort_values()
        for r_m, r_km in zip(RADII_M, RADII_KM):
            radius_maps[r_m][site] = ordered[ordered <= r_km].index.tolist()

    # ── hourly real-sensor traffic wide matrix ────────────────────────────────
    df_tc = df_traffic.copy()
    df_tc = df_tc[df_tc["type"] == "FIETSERS"]
    df_tc["count"]    = df_tc["count"].fillna(0)
    df_tc["site_id"]  = df_tc["site_id"].astype("int32")
    df_tc["datehour"] = _pd.to_datetime(df_tc["from"]).dt.floor("h")
    df_hourly = (df_tc.groupby(["site_id","datehour"], as_index=False)["count"]
                 .sum().rename(columns={"count":"traffic_total"}))
    traffic_wide = (df_hourly
                    .pivot_table(index="datehour", columns="site_id",
                                 values="traffic_total", aggfunc="sum")
                    .sort_index())

    # ── 24 target hours (memory key: slice BEFORE site loop) ─────────────────
    target_hours = _pd.date_range(predict_date, periods=24, freq="h")

    lag_dfs = []
    for LAG in [1, 24, 168]:
        tw_lag = traffic_wide.shift(LAG)

        # *** filter to prediction date rows right here — 24 rows, not 1000+ ***
        tw_lag_pred = tw_lag[tw_lag.index.isin(target_hours)]
        if tw_lag_pred.empty:
            continue

        nan_tpl = _pd.Series(_np.nan, index=tw_lag_pred.index)
        parts = []
        for site in new_ids:
            row = {"site_id": site, "datehour": tw_lag_pred.index}
            for r_m in RADII_M:
                rad_cols = [s for s in radius_maps[r_m][site]
                            if s in tw_lag_pred.columns]
                pfx = f"nb_r{r_m}m_lag{LAG}h"
                if rad_cols:
                    sub = tw_lag_pred[rad_cols]
                    row[f"{pfx}_mean"] = sub.mean(axis=1).values
                    row[f"{pfx}_max"]  = sub.max(axis=1).values
                    row[f"{pfx}_sum"]  = sub.sum(axis=1).values
                    row[f"{pfx}_std"]  = sub.std(axis=1).values
                else:
                    for s in ("mean","max","sum","std"):
                        row[f"{pfx}_{s}"] = nan_tpl.values
            parts.append(_pd.DataFrame(row))
        lag_dfs.append(_pd.concat(parts, ignore_index=True))

    if not lag_dfs:
        raise ValueError(f"No lag data found for predict_date={predict_date}. "
                         "Check that 2026-02 and 2026-03 traffic files are loaded.")

    df_out = lag_dfs[0]
    for df_lag in lag_dfs[1:]:
        df_out = df_out.merge(df_lag, on=["site_id","datehour"], how="outer")
    df_out.columns = [c.replace("168h","7d") for c in df_out.columns]
    df_out = df_out[["site_id","datehour"] + [x for x in df_out if x.startswith("nb")]]
    df_out = df_out.loc[:, df_out.isna().mean() <= 0.7]
    return df_out


def compute_neighbor_traffic_lag_15m_features_cross(
        df_traffic, df_site_real, df_site_new, predict_date=PREDICT_DATE):
    """15-minute lag version — same memory optimisation as the hourly version."""
    import numpy as _np
    import pandas as _pd

    RADII_M  = [200, 500, 1000, 2000]
    RADII_KM = [r / 1000 for r in RADII_M]

    real = (df_site_real[["site_id","latitude","longtitude"]]
            .drop_duplicates("site_id").sort_values("site_id"))
    real_ids = real["site_id"].to_numpy()
    real_lat  = real["latitude"].to_numpy()
    real_lon  = real["longtitude"].to_numpy()

    new = (df_site_new[["site_id","latitude","longtitude"]]
           .drop_duplicates("site_id").sort_values("site_id"))
    new_ids = new["site_id"].to_numpy()
    new_lat  = new["latitude"].to_numpy()
    new_lon  = new["longtitude"].to_numpy()

    R = 6371.0
    nl = _np.radians(new_lat);  nlo = _np.radians(new_lon)
    rl = _np.radians(real_lat); rlo = _np.radians(real_lon)
    dlat = nl[:, None] - rl[None, :]
    dlon = nlo[:, None] - rlo[None, :]
    a    = _np.sin(dlat/2)**2 + _np.cos(nl)[:,None]*_np.cos(rl)[None,:]*_np.sin(dlon/2)**2
    dist = 2 * R * _np.arcsin(_np.sqrt(a))
    dist_df = _pd.DataFrame(dist, index=new_ids, columns=real_ids)

    radius_maps = {r_m: {} for r_m in RADII_M}
    for site in new_ids:
        ordered = dist_df.loc[site].sort_values()
        for r_m, r_km in zip(RADII_M, RADII_KM):
            radius_maps[r_m][site] = ordered[ordered <= r_km].index.tolist()

    df_tc = df_traffic.copy()
    df_tc = df_tc[df_tc["type"] == "FIETSERS"]
    df_tc["count"]   = df_tc["count"].fillna(0)
    df_tc["site_id"] = df_tc["site_id"].astype("int32")
    df_tc["from"]    = _pd.to_datetime(df_tc["from"])

    df_15min = (df_tc.groupby(["site_id","from"], as_index=False)["count"]
                .sum().rename(columns={"count":"total_fiets_15min"}))
    tw15 = (df_15min
            .pivot_table(index="from", columns="site_id",
                         values="total_fiets_15min", aggfunc="sum")
            .sort_index())
    tw15.index.name = "ts"
    tw15_shifted = tw15.shift(1)

    # on-the-hour rows only (one row per hour)
    tw_prev15 = tw15_shifted[tw15_shifted.index.minute == 0].copy()
    tw_prev15.index.name = "datehour"

    # *** filter to prediction date BEFORE the site loop ***
    target_hours = _pd.date_range(predict_date, periods=24, freq="h")
    tw_prev15_pred = tw_prev15[tw_prev15.index.isin(target_hours)]

    if tw_prev15_pred.empty:
        raise ValueError(f"No 15m data found for predict_date={predict_date}.")

    nan_tpl = _pd.Series(_np.nan, index=tw_prev15_pred.index)
    parts = []
    for site in new_ids:
        row = {"site_id": site, "datehour": tw_prev15_pred.index}
        for r_m in RADII_M:
            rad_cols = [s for s in radius_maps[r_m][site]
                        if s in tw_prev15_pred.columns]
            pfx = f"nb_r{r_m}m_lag15m"
            if rad_cols:
                sub = tw_prev15_pred[rad_cols]
                row[f"{pfx}_mean"] = sub.mean(axis=1).values
                row[f"{pfx}_max"]  = sub.max(axis=1).values
                row[f"{pfx}_sum"]  = sub.sum(axis=1).values
                row[f"{pfx}_std"]  = sub.std(axis=1).values
            else:
                for s in ("mean","max","sum","std"):
                    row[f"{pfx}_{s}"] = nan_tpl.values
        parts.append(_pd.DataFrame(row))

    df_out = _pd.concat(parts, ignore_index=True)
    df_out = df_out[["site_id","datehour"] + [x for x in df_out if x.startswith("nb")]]
    df_out = df_out.loc[:, df_out.isna().mean() <= 0.7]
    return df_out

######## WEATHER CRAWL FUNCTION ########

def crawl_weather_for_location(
        lat, lon,
        start_date, end_date,
        site_id=1,
        cache_path=None,
        archive_cutoff="2026-03-15",
        hourly_variables=None,
        max_retries=5,
        delay_between_chunks=2.0):
    """
    Crawl hourly weather data for a single (lat, lon) from Open-Meteo.

    - Uses Archive API  for dates <= archive_cutoff (historical data)
    - Uses Forecast API for dates >  archive_cutoff (recent / future)
    - Handles 429 rate-limits with exponential backoff + jitter
    - Saves result to cache_path (CSV) so re-running the cell is instant

    Returns a DataFrame with columns:
      [site_id, latitude, longitude, datetime,
       temperature_2m, relative_humidity_2m, precipitation,
       rain, snowfall, wind_speed_10m, wind_direction_10m,
       pressure_msl, cloud_cover]
    which matches the format of site_id_weather_output_new.csv.
    """
    import requests as _req
    import time     as _time
    import random   as _rnd
    import os       as _os
    from datetime import date as _date

    # ── load from cache if available ──────────────────────────────────────────
    if cache_path and _os.path.exists(cache_path):
        df_cache = pd.read_csv(cache_path)
        print(f"  Weather loaded from cache: {cache_path}  ({len(df_cache):,} rows)")
        return df_cache

    ARCHIVE_URL  = "https://archive-api.open-meteo.com/v1/archive"
    FORECAST_URL = "https://api.open-meteo.com/v1/forecast"

    if hourly_variables is None:
        hourly_variables = [
            "temperature_2m", "relative_humidity_2m",
            "precipitation", "rain", "snowfall",
            "wind_speed_10m", "wind_direction_10m",
            "pressure_msl", "cloud_cover",
        ]

    # ── normalise date types ──────────────────────────────────────────────────
    if isinstance(start_date,      str): start_date      = _date.fromisoformat(start_date)
    if isinstance(end_date,        str): end_date        = _date.fromisoformat(end_date)
    if isinstance(archive_cutoff,  str): archive_cutoff  = _date.fromisoformat(archive_cutoff)

    # ── split into year chunks (archive vs forecast) ──────────────────────────
    chunks = []
    cur = start_date
    while cur <= end_date:
        chunk_end = min(_date(cur.year, 12, 31), end_date)
        if cur <= archive_cutoff:
            api_end = min(chunk_end, archive_cutoff)
            chunks.append(("archive",  cur,             api_end))
            if chunk_end > archive_cutoff:
                chunks.append(("forecast", archive_cutoff, chunk_end))
        else:
            chunks.append(("forecast", cur, chunk_end))
        cur = _date(cur.year + 1, 1, 1)

    # ── crawl each chunk ──────────────────────────────────────────────────────
    all_rows = []
    for api_type, c_start, c_end in chunks:
        url    = ARCHIVE_URL if api_type == "archive" else FORECAST_URL
        params = {
            "latitude":   lat,
            "longitude":  lon,
            "start_date": c_start.strftime("%Y-%m-%d"),
            "end_date":   c_end.strftime("%Y-%m-%d"),
            "hourly":     ",".join(hourly_variables),
            "timezone":   "Europe/Brussels",
        }
        print(f"    [{api_type:>8}] {c_start} → {c_end} ...", end=" ", flush=True)

        data = None
        for attempt in range(max_retries):
            try:
                resp = _req.get(url, params=params, timeout=60)
                if resp.status_code == 429:
                    wait = min(300, 30 * (2 ** attempt)) + _rnd.uniform(0, 10)
                    print(f"\n      [429] rate-limited — waiting {wait:.0f}s (attempt {attempt+1}/{max_retries}) ...",
                          end=" ", flush=True)
                    _time.sleep(wait)
                    continue
                resp.raise_for_status()
                data = resp.json()
                break
            except Exception as exc:
                if attempt < max_retries - 1:
                    wait = 30 + _rnd.uniform(0, 10)
                    print(f"\n      [Error] {exc} — retrying in {wait:.0f}s ...",
                          end=" ", flush=True)
                    _time.sleep(wait)
                else:
                    print(f"FAILED: {exc}")

        if data is None:
            continue

        hourly = data.get("hourly", {})
        times  = hourly.get("time",  [])
        for j, ts in enumerate(times):
            row = {
                "site_id":   site_id,
                "latitude":  lat,
                "longitude": lon,
                "datetime":  ts,
            }
            for var in hourly_variables:
                vals    = hourly.get(var, [])
                row[var] = vals[j] if j < len(vals) else None
            all_rows.append(row)

        print(f"OK — {len(times):,} records")
        _time.sleep(delay_between_chunks)

    df_out = pd.DataFrame(all_rows)

    if cache_path:
        df_out.to_csv(cache_path, index=False)
        print(f"  Weather saved to cache: {cache_path}  ({len(df_out):,} rows)")

    return df_out

# PREDICTION — Leuven Bike Roads on 2026-03-01 (24 h)

Predicts hourly bike traffic for all new Leuven road segments in `leuven_sites.csv`
for a single day (2026-03-01, hours 0-23) using the trained `model_final.pickle`.

**Run cells below independently** — they do not require the training cells above.

## P1  Load Sites

In [4]:
# Real sensor sites (traffic source for lag features)
df_site_real = pd.read_csv(FILEPATH_SITE, names=SITE_COLS)
print(f"Real sensor sites : {len(df_site_real):,}")

# New Leuven road segments (prediction targets)
df_site_new = pd.read_csv(FILEPATH_LEUVEN_SITE, names=SITE_COLS)
print(f"Leuven road sites : {len(df_site_new):,}")

leuven_ids  = df_site_new["site_id"].to_numpy()
hours_pred  = pd.date_range(PREDICT_DATE, periods=24, freq="h")
print(f"Prediction hours  : {hours_pred[0]}  →  {hours_pred[-1]}")

Real sensor sites : 151


FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\Admin\\Desktop\\KU Leuven\\Sem 2\\MDA\\mda_project\\mda_project\\explo\\andy\\explo/andy/output/leuven_sites.csv'

## P2  Neighbor Traffic Features (hourly lags)

In [ ]:
# Derive which traffic months to load from PREDICT_DATE:
#   - current month  (contains the prediction day)
#   - previous month (needed for lag-7d: 7 days before the prediction date)
_pred_dt     = pd.Timestamp(PREDICT_DATE)
_prev_month  = (_pred_dt - pd.DateOffset(days=7)).strftime("%Y-%m")
_curr_month  = _pred_dt.strftime("%Y-%m")
lmonth = list(dict.fromkeys([_prev_month, _curr_month]))  # deduplicated, ordered
print(f"Traffic months to load: {lmonth}")

df_traffic_real = pd.concat([
    pd.read_csv(PATH_TRAFFIC + f"data-{m}.csv", names=TRAFFIC_COLS)
    for m in lmonth
], ignore_index=True)
print(f"  Raw rows: {len(df_traffic_real):,}")

df_fts_neighbor_lag = compute_neighbor_traffic_lag_hour_features_cross(
    df_traffic   = df_traffic_real,
    df_site_real = df_site_real,
    df_site_new  = df_site_new,
)
df_fts_neighbor_lag = df_fts_neighbor_lag[
    df_fts_neighbor_lag["datehour"].dt.strftime("%Y-%m-%d") == PREDICT_DATE
].reset_index(drop=True)

print(f"Neighbor lag features shape: {df_fts_neighbor_lag.shape}")
df_fts_neighbor_lag.head(3)

## P3  Neighbor Traffic Features (15-min lag)

In [ ]:
print(f"Computing 15-min neighbor features for {len(df_site_new):,} Leuven sites ...")
df_fts_neighbor_lag15m = compute_neighbor_traffic_lag_15m_features_cross(
    df_traffic   = df_traffic_real,   # already loaded in P2
    df_site_real = df_site_real,
    df_site_new  = df_site_new,
)
df_fts_neighbor_lag15m = df_fts_neighbor_lag15m[
    df_fts_neighbor_lag15m["datehour"].dt.strftime("%Y-%m-%d") == PREDICT_DATE
].reset_index(drop=True)

print(f"Neighbor 15m features shape: {df_fts_neighbor_lag15m.shape}")
df_fts_neighbor_lag15m.head(3)

## P4  Weather Features

In [ ]:
# ── Weather for Leuven ────────────────────────────────────────────────────────
# All Leuven segments are within ~57 km², so one representative location
# captures the weather for the whole city.
# Start 2 days before prediction date to cover 24h rolling-lag warm-up.

LEUVEN_CENTER_LAT = 50.8798   # approx. centre of Leuven
LEUVEN_CENTER_LON = 4.7005

# Dynamic: 2 days before PREDICT_DATE for rolling-24h lag context
_wt_start = (pd.Timestamp(PREDICT_DATE) - pd.Timedelta(days=2)).strftime("%Y-%m-%d")
_date_tag  = PREDICT_DATE.replace("-", "_")
wt_cache   = os.path.join(
    os.path.dirname(FILEPATH_LEUVEN_SITE),
    f"leuven_weather_{_date_tag}.csv"   # date-stamped cache
)

print(f"Fetching Leuven weather ({_wt_start} → {PREDICT_DATE}) ...")
df_wt_raw = crawl_weather_for_location(
    lat        = LEUVEN_CENTER_LAT,
    lon        = LEUVEN_CENTER_LON,
    start_date = _wt_start,
    end_date   = PREDICT_DATE,
    site_id    = 1,
    cache_path = wt_cache,
)
print(f"Raw weather rows: {len(df_wt_raw)}")

# ── Compute lag / rolling weather features ────────────────────────────────────
df_wt_features = compute_weather_features(df_weather=df_wt_raw)
df_wt_features["datehour"] = pd.to_datetime(df_wt_features["datehour"])

wt_cols = [c for c in df_wt_features.columns if c.startswith("wt_")]
df_wt_pred = (
    df_wt_features[df_wt_features["datehour"].dt.strftime("%Y-%m-%d") == PREDICT_DATE]
    [["datehour"] + wt_cols]
    .reset_index(drop=True)
)
print(f"Weather feature rows for {PREDICT_DATE}: {len(df_wt_pred)}")

# ── Broadcast to ALL Leuven sites ─────────────────────────────────────────────
df_fts_weather_lag = pd.DataFrame({
    "site_id"  : np.repeat(leuven_ids, 24),
    "datehour" : np.tile(hours_pred,   len(leuven_ids)),
}).merge(df_wt_pred, on="datehour", how="left")

print(f"Weather feature shape: {df_fts_weather_lag.shape}")
df_fts_weather_lag.head(3)

## P5  Population Features

In [ ]:
# This reads the GeoTIFF and may take a few minutes for 7,698 sites.
# If previously computed, load from cache instead.
pop_cache = os.path.join(os.path.dirname(FILEPATH_LEUVEN_SITE), "leuven_population_features.csv")
if os.path.exists(pop_cache):
    df_fts_population = pd.read_csv(pop_cache)
    print(f"Population features loaded from cache: {df_fts_population.shape}")
else:
    print("Computing population features (this may take a few minutes)...")
    df_fts_population = compute_population_features(
        df_site  = df_site_new,
        tif_path = FILEPATH_POP_TIF,
    )
    df_fts_population.to_csv(pop_cache, index=False)
    print(f"Population features saved to cache: {df_fts_population.shape}")

df_fts_population.head(3)

## P6  Datetime Features

In [ ]:
# Build a synthetic skeleton: all Leuven sites × 24 hours of the prediction date
import holidays

dt = pd.DataFrame({
    "site_id"  : np.repeat(leuven_ids, 24),
    "datehour" : np.tile(hours_pred,   len(leuven_ids)),
})
dt["year"]      = dt["datehour"].dt.year
dt["month"]     = dt["datehour"].dt.month
dt["day"]       = dt["datehour"].dt.day
dt["hour"]      = dt["datehour"].dt.hour
dt["dayofweek"] = dt["datehour"].dt.dayofweek

dt["dt_is_weekend"]       = dt["dayofweek"].isin([5, 6]).astype(int)
dt["dt_is_weekday"]       = 1 - dt["dt_is_weekend"]
dt["dt_is_monday"]        = (dt["dayofweek"] == 0).astype(int)
dt["dt_is_friday"]        = (dt["dayofweek"] == 4).astype(int)
dt["dt_is_morning_rush"]  = dt["hour"].isin([7, 8, 9]).astype(int)
dt["dt_is_afternoon_rush"]= dt["hour"].isin([16, 17, 18]).astype(int)
dt["dt_is_rush_hour"]     = (dt["dt_is_morning_rush"] | dt["dt_is_afternoon_rush"]).astype(int)
dt["dt_is_lunch_hour"]    = dt["hour"].isin([12, 13]).astype(int)
dt["dt_is_night"]         = dt["hour"].isin([0, 1, 2, 3, 4, 5]).astype(int)
dt["dt_is_business_hours"]= (dt["hour"].between(9, 17) & (dt["dt_is_weekday"] == 1)).astype(int)

bel_holidays = list(holidays.Belgium(years=range(2024, 2027)).keys())
dt["dt_is_holiday"] = dt["datehour"].dt.normalize().isin(bel_holidays).astype(int)

df_fts_datetime = dt[["site_id", "datehour", "year", "month", "day", "hour", "dayofweek"]
                      + [c for c in dt if c.startswith("dt_")]]

print(f"Datetime feature shape: {df_fts_datetime.shape}")
df_fts_datetime.head(3)

## P7  Site Features

In [ ]:
# Bike roads never start with T/N/R/A road codes → all flags = 0
df_fts_site = pd.DataFrame({
    "site_id"               : np.repeat(leuven_ids, 24),
    "datehour"              : np.tile(hours_pred,   len(leuven_ids)),
    "site_is_road_tunnel"   : 0,
    "site_is_road_national" : 0,
    "site_is_road_ring"     : 0,
    "site_is_road_motorway" : 0,
})
print(f"Site feature shape: {df_fts_site.shape}")
df_fts_site.head(3)

## P8  Infrastructure & POI Features

In [ ]:
df_fts_infra = compute_infra_features(
    df_infra = pd.read_csv(FILEPATH_LEUVEN_INFRA),   # leuven_bike_roads.csv
    df_site  = df_site_new,                           # leuven_sites.csv
)
print(f"Infra feature shape: {df_fts_infra.shape}")
df_fts_infra.head(3)

## P9  Merge All Features

In [ ]:
# Base: all Leuven sites × 24 hours
df_base = pd.DataFrame({
    "site_id"  : np.repeat(leuven_ids, 24),
    "datehour" : np.tile(hours_pred,   len(leuven_ids)),
})
df_base["datehour"] = pd.to_datetime(df_base["datehour"])

df_pred = (df_base
    .merge(df_fts_neighbor_lag,    on=["site_id", "datehour"], how="left")
    .merge(df_fts_neighbor_lag15m, on=["site_id", "datehour"], how="left")
    .merge(df_fts_datetime,        on=["site_id", "datehour"], how="left")
    .merge(df_fts_population,      on="site_id",               how="left")
    .merge(df_fts_weather_lag,     on=["site_id", "datehour"], how="left")
    .merge(df_fts_site,            on=["site_id", "datehour"], how="left")
    .merge(df_fts_infra,           on="site_id",               how="left")
)

print(f"Merged prediction frame: {df_pred.shape}")
print(f"  site_id range : {df_pred.site_id.min()} – {df_pred.site_id.max()}")
print(f"  datehour range: {df_pred.datehour.min()} – {df_pred.datehour.max()}")
df_pred.head(3)

## P10  Load Model & Predict

In [ ]:
import pickle

with open(FILEPATH_MODEL, "rb") as f:
    model_final = pickle.load(f)

# The model expects the exact feature columns it was trained on
FEATURE_COLS_MODEL = list(model_final.feature_names_in_)
print(f"Model expects {len(FEATURE_COLS_MODEL)} features")

# Check which features are present vs missing
present  = [c for c in FEATURE_COLS_MODEL if c in df_pred.columns]
missing  = [c for c in FEATURE_COLS_MODEL if c not in df_pred.columns]
print(f"  Present : {len(present)}")
print(f"  Missing : {len(missing)}")
if missing:
    print(f"  Missing columns (will be filled NaN): {missing[:10]}{'...' if len(missing)>10 else ''}")

# Build X — fill any missing columns with NaN (LightGBM handles NaN natively)
X_pred = pd.DataFrame(index=df_pred.index)
for col in FEATURE_COLS_MODEL:
    X_pred[col] = df_pred[col] if col in df_pred.columns else np.nan

# Predict (clip to >= 0)
y_pred = np.clip(model_final.predict(X_pred), 0, None)

df_result = df_pred[["site_id", "datehour"]].copy()
df_result["predicted_traffic"] = y_pred.round(2)

print(f"\nPrediction summary:")
print(df_result["predicted_traffic"].describe().round(2))
df_result.head(10)

## P11  Export Predictions

In [ ]:
# Merge back site metadata (road name, lat, lon) for readability
site_meta = df_site_new.rename(columns={
    "site_id"  : "site_id",
    "site_nr"  : "osmid",
    "longtitude": "longitude",
    "name"     : "road_name",
})[["site_id", "osmid", "longitude", "latitude", "road_name"]]

df_export = df_result.merge(site_meta, on="site_id", how="left")[[
    "site_id", "osmid", "road_name", "longitude", "latitude",
    "datehour", "predicted_traffic"
]]

df_export.to_csv(OUTPUT_PRED_PATH, index=False)
print(f"Saved: {OUTPUT_PRED_PATH}")
print(f"  {len(df_export):,} rows  ({df_export['site_id'].nunique():,} sites x 24 hours)")
print()
print("Preview:")
display(df_export.pivot_table(index="road_name", columns="datehour",
                               values="predicted_traffic", aggfunc="mean")
        .round(1).head(10))